# Restoring organs, run 3b: N=12 packing with the diagnosed fixes (tag `pack3x4_v2`)

Run 3 collapsed at N=12. Local diagnosis found two causes:
1. **Wrong fence.** When a neighbour was missing or displaced, the fence search returned the face one cell further
   back and the push seated the cube into the hole (errors of exactly 40 mm, one pitch). The restore now rejects any
   fence more than half a pitch short of the target cell and stops at the nominal position.
2. **Knocked-out finished cubes.** Row-2 restores pushed the row-1 end cube from 3.7 mm to 13 mm and nothing re-seated
   it. `restore_reseat` re-seats up to two finished cubes that a stage pushed out of tolerance.

Local scripted check, N=12, 5 mm placement error: final block correct in 3/3 chains, all 36 cubes within 8 mm,
collateral at most 7 mm.

**Arms:** `restore_v2` (fix 1) and `restore_reseat` (fixes 1 + 2), N in {2, 4, 8, 12}, 6 seeds. Same trained policy
(`dp_pack3x4`), demos and seeds as run 3, so the last cell compares pairwise against run 3's none / noop / restore.

**Runtime on a T4: about 2 h** (demo and train cells skip because run 3's files are complete). Resumable per chain.

In [ ]:
import os, subprocess, sys
os.environ["MUJOCO_GL"] = "egl"
os.environ["USE_FLAX"] = "0"   # Colab ships jax+flax; diffusers 0.11.1 flax schedulers use the removed jax.random.KeyArray
def pip(*args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", *args], capture_output=True, text=True)
    tail = (r.stdout + r.stderr)[-2500:]
    print(tail)
    if r.returncode != 0:
        raise SystemExit(f"pip failed on {args} -- paste the output above")
# robosuite 1.5.2 requires mujoco>=3.3.0; 3.3.7 is the newest that still has MjData.qM (needed by its OSC controller).
# It also pulls mink==0.0.5, which pins numpy<2 -- and numpy 1.x has no Python 3.13 wheels. mink is only used for
# GR1 whole-body IK and robosuite guards its import, so install robosuite without deps + the runtime deps we need.
pip("numpy>=2", "mujoco==3.3.7", "h5py", "glfw", "pynput", "numba", "scipy", "Pillow", "opencv-python", "termcolor", "tqdm")
pip("--no-deps", "robosuite==1.5.2")
# PyPI robomimic is a 2023 sdist; install master without deps, then the deps we actually use.
# diffusers==0.11.1 + huggingface_hub==0.23.4 is robomimic's own pinned pair (EMAModel(model=...) API).
pip("--no-deps", "git+https://github.com/ARISE-Initiative/robomimic.git")
# diffusers 0.11.1 auto-imports transformers when present; Colab's transformers 5.x needs huggingface_hub>=1.5
# while diffusers 0.11.1 needs <=0.23. They cannot coexist and nothing here uses transformers, so remove it.
r = subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "transformers"], capture_output=True, text=True)
print(r.stdout[-200:] or r.stderr[-200:])
pip("diffusers==0.11.1", "huggingface_hub==0.23.4", "torchvision", "tensorboard", "tensorboardX",
    "imageio", "imageio-ffmpeg", "psutil", "matplotlib")
import mujoco, robosuite, robomimic, torch, diffusers, numpy, logging
logging.getLogger("robosuite_logs").setLevel(logging.ERROR)   # silence per-reset controller-config warnings
print("mujoco", mujoco.__version__, "| robosuite", robosuite.__version__, "| robomimic", robomimic.__version__,
      "| diffusers", diffusers.__version__, "| numpy", numpy.__version__, "| cuda", torch.cuda.is_available())
assert mujoco.__version__.startswith("3.3"), "wrong mujoco: robosuite's OSC needs MjData.qM (removed later)"
from robomimic.algo import diffusion_policy                      # fails fast if the diffusers pin is wrong
# early OSC probe: this is exactly what breaks on an incompatible mujoco
from robosuite.controllers import load_composite_controller_config
env = robosuite.make("Lift", robots="Panda", controller_configs=load_composite_controller_config(controller="BASIC", robot="Panda"),
                     has_renderer=False, has_offscreen_renderer=False, use_camera_obs=False, control_freq=20, horizon=10)
env.reset(); env.step(env.action_spec[0] * 0); env.close()
print("robosuite OSC probe OK -- environment is good")

In [ ]:
# ---- Drive: mount, then PROVE the mount is real before touching any Drive path ----
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/restoring_organs"      # checkpoints, demos, mirrored results
RES = "/content/results"                              # results are written here FIRST, then mirrored to Drive
KNOWN = f"{OUT}/demos.hdf5"                           # a file that is known to exist on a real mount
import os, json, pathlib, time, shutil, numpy as np
if not os.path.isfile(KNOWN):
    raise SystemExit(f"Drive is NOT really mounted ({KNOWN} not found). Run drive.mount('/content/drive', force_remount=True), "
                     "then re-run this cell. Nothing was written.")
os.makedirs(RES, exist_ok=True)

def drive_ok():
    return os.path.isfile(KNOWN)

def mirror(name):
    # copy RES/name to Drive; never fatal, never a rename or delete on Drive
    try:
        if not drive_ok():
            print("WARNING: Drive not mounted, local copy only:", name); return False
        shutil.copyfile(f"{RES}/{name}", f"{OUT}/{name}"); return True
    except Exception as ex:
        print("WARNING: Drive mirror failed for", name, repr(ex), "(local copy kept)"); return False

def save(name, obj):
    tmp = f"{RES}/{name}.tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=1)
    os.replace(tmp, f"{RES}/{name}")                  # local disk only
    mirror(name)

def load(name):
    # the longer of the local and Drive copies: a fresh VM only has Drive, an unmirrored save only has local
    best = None
    for p in (f"{RES}/{name}", f"{OUT}/{name}"):
        try:
            v = json.load(open(p))
        except Exception:
            continue
        if best is None or len(v) > len(best):
            best = v
    return best

print("Drive mount verified via", KNOWN)
print("Drive folder:", sorted(p.name for p in pathlib.Path(OUT).iterdir()))

In [ ]:
%%writefile rs_chain.py
"""3-D line-up chain in robosuite (Panda + parallel gripper, MuJoCo).

Same experiment as the planar rig, with real grasping:
  stage k: pick cube k from staging, place it at a target x measured relative to
           where cube k-1 ACTUALLY landed (error propagates down the row).
  arms:    none / noop (matched duration, touches nothing) / restore
  restore: closed fingers as a plate; re-pick if out of zone, seat against the rail, push into the fence
           in -x to contact with cube k-1 (cube 0 against a fixed stop).
Requires robosuite==1.5.2 with mujoco==3.2.7 (3.12 removed MjData.qM).
"""
import numpy as np
from robosuite.environments.manipulation.manipulation_env import ManipulationEnv
from robosuite.models.arenas import TableArena
from robosuite.models.objects import BoxObject
from robosuite.models.tasks import ManipulationTask
from robosuite.utils.mjcf_utils import array_to_string
from robosuite.controllers import load_composite_controller_config

CUBE_H = 0.020                     # half-size -> contact pitch 0.040
PITCH = 2 * CUBE_H
TABLE_Z = 0.80
CUBE_Z = TABLE_Z + CUBE_H
RAIL_Y, RAIL_HY, RAIL_HZ = -0.12, 0.008, 0.015   # 3 cm rail: a pushed cube tilts over anything lower (measured 8-14 mm spring-back); recovery grasps use x-jaws so no finger crosses it
RAIL_FACE = RAIL_Y + RAIL_HY
ROW_Y = RAIL_FACE + CUBE_H          # cube y when seated on the rail
STOP_X, STOP_HX = -0.26, 0.008
STOP_FACE = STOP_X + STOP_HX
X0 = STOP_FACE + CUBE_H             # cube 0 when flush against the stop
STAGE_Y = 0.20
HOVER = 0.10


class LineupEnv(ManipulationEnv):
    def __init__(self, n_cubes=8, seed=0):
        self.n_cubes = n_cubes
        self.table_full_size = (0.8, 0.8, 0.05)
        self.table_friction = (1.0, 5e-3, 1e-4)
        self.table_offset = np.array((0, 0, TABLE_Z))
        cfg = load_composite_controller_config(controller="BASIC", robot="Panda")
        super().__init__(robots="Panda", controller_configs=cfg, base_types="default",
                         gripper_types="default", initialization_noise=None,
                         use_camera_obs=False, has_renderer=False, has_offscreen_renderer=False,
                         control_freq=20, horizon=100000, ignore_done=True, hard_reset=False,
                         seed=seed)

    def reward(self, action=None):
        return 0.0

    def _load_model(self):
        super()._load_model()
        xpos = self.robots[0].robot_model.base_xpos_offset["table"](self.table_full_size[0])
        self.robots[0].robot_model.set_base_xpos(xpos)
        arena = TableArena(table_full_size=self.table_full_size, table_friction=self.table_friction,
                           table_offset=self.table_offset)
        arena.set_origin([0, 0, 0])
        self.cubes = [BoxObject(name=f"cube{i}", size=[CUBE_H] * 3, rgba=[0.85, 0.45, 0.2, 1])
                      for i in range(self.n_cubes)]
        self.rail = BoxObject(name="rail", size=[0.32, RAIL_HY, RAIL_HZ], rgba=[0.45, 0.5, 0.55, 1], joints=None)
        self.rail.get_obj().set("pos", array_to_string([0.0, RAIL_Y, TABLE_Z + RAIL_HZ]))
        self.stop = BoxObject(name="stop", size=[STOP_HX, 0.05, CUBE_H], rgba=[0.45, 0.5, 0.55, 1], joints=None)
        self.stop.get_obj().set("pos", array_to_string([STOP_X, ROW_Y, CUBE_Z]))
        self.model = ManipulationTask(mujoco_arena=arena, mujoco_robots=[self.robots[0].robot_model],
                                      mujoco_objects=self.cubes + [self.rail, self.stop])

    def _setup_references(self):
        super()._setup_references()
        self.cube_bid = [self.sim.model.body_name2id(c.root_body) for c in self.cubes]

    def _reset_internal(self):
        super()._reset_internal()
        for i, c in enumerate(self.cubes):
            self.set_cube(i, [-0.22 + 0.06 * i, STAGE_Y, CUBE_Z])

    # ---- helpers ----
    def set_cube(self, i, pos):
        self.sim.data.set_joint_qpos(self.cubes[i].joints[0],
                                     np.concatenate([np.asarray(pos, float), [1, 0, 0, 0]]))
        self.sim.forward()

    def cube_pos(self, i):
        return np.array(self.sim.data.body_xpos[self.cube_bid[i]])


class Chain:
    """Expert, restoring organ, control, and chain runner over a LineupEnv."""

    def __init__(self, n_cubes=8, seed=0):
        self.env = LineupEnv(n_cubes=n_cubes, seed=seed)
        self.rng = np.random.default_rng(seed)
        self.obs = self.env.reset()
        self.steps_used = 0
        self.rec = None          # list of (obs_dict, action) while recording demos
        self.cur_i = 0
        self.cur_target = np.zeros(3)
        self.q_home = np.asarray(self.obs["robot0_eef_quat"], float).copy()   # straight-down wrist: the reference for every rotation

    def reseed(self, seed):
        """Reuse ONE env across runs: robosuite/MuJoCo do not free envs, and
        constructing one per chain exhausts resources after ~18. Only the error
        RNG needs reseeding; the task has no other randomness."""
        self.rng = np.random.default_rng(seed); self.steps_used = 0
        return self

    # ---- policy-facing observation (identical for demos and rollouts) ----
    def obs_dict(self):
        return {'eef_pos': np.array(self.obs['robot0_eef_pos'], np.float32),
                'gripper_qpos': np.array(self.obs['robot0_gripper_qpos'], np.float32),
                'cube_pos': self.env.cube_pos(self.cur_i).astype(np.float32),
                'target': np.asarray(self.cur_target, np.float32)}

    def _step(self, a):
        if self.rec is not None:
            self.rec.append((self.obs_dict(), np.array(a, np.float32)))
        self.obs, _, _, _ = self.env.step(a)
        self.steps_used += 1

    # ---- low level ----
    def eef(self):
        return np.array(self.obs["robot0_eef_pos"])

    def goto(self, target, grip, steps, gain=8.0):
        target = np.asarray(target, float)
        for _ in range(steps):
            a = np.zeros(self.env.action_dim)
            a[:3] = np.clip(gain * (target - self.eef()), -1, 1)
            a[-1] = grip
            self._step(a)
        return self.eef()

    def push_axis(self, axis, target, hold, grip, max_steps, amp=0.7, tol=0.0015, cube=None, c_stall_n=5, z_bias=0.0, cube_target=None):
        """Saturated push along one axis (the OSC goal sits a fixed distance ahead,
        so the commanded force is ~kp*amp*0.05 regardless of remaining error), other
        axes held by P-control. Stops on arrival or when contact stalls progress.
        A P-controller cannot do this: its force vanishes as it approaches."""
        hold = np.asarray(hold, float)
        amp = amp * getattr(self, "amp_scale", 1.0)          # retry-at-lower-force hook (1.0 unless a subclass sets it)
        last = self.eef()[axis]; stall = 0
        c0 = self.env.cube_pos(cube)[axis] if cube is not None else None; c_last = c0; c_stall = 0
        # optional collateral guard (off unless a subclass sets self.watch): snapshot the finished parts and abort the
        # push the moment any of them moves, instead of shoving a seated block. Also a sign-crossing arrival test, since
        # a cube driven at full amplitude can jump the +/-1.5 mm window in one control step.
        watch = [j for j in (getattr(self, "watch", None) or []) if j != cube]
        w0 = {j: self.env.cube_pos(j).copy() for j in watch}
        push_sign = np.sign(target - self.eef()[axis]) or 1.0
        for _ in range(max_steps):
            e = self.eef()
            d = target - e[axis]
            if abs(d) < tol:
                break
            if cube_target is not None and abs(self.env.cube_pos(cube)[axis] - cube_target) < 0.0015:   # the cube is seated: stop before overdriving it over the fence
                break
            if cube_target is not None and push_sign * (self.env.cube_pos(cube)[axis] - cube_target) >= -0.0015:
                break                # crossed the target (a full-amplitude push can jump the +/-1.5 mm window above)
            if watch and max(np.abs(self.env.cube_pos(j)[:2] - w0[j][:2]).max() for j in watch) > 0.0015:
                self.collateral_aborts = getattr(self, "collateral_aborts", 0) + 1
                break
            a = np.zeros(self.env.action_dim)
            a[:3] = np.clip(8.0 * (hold - e), -1, 1)
            a[axis] = amp * np.sign(d)
            a[2] = float(np.clip(a[2] + z_bias, -1, 1))
            a[-1] = grip
            self._step(a)
            stall = stall + 1 if abs(e[axis] - last) < 1e-4 else 0
            last = e[axis]
            if stall >= 14:         # arrested by contact (loose: slow friction creep is not a stall)
                break
            if cube is not None:    # the pushed cube itself has been arrested by a fence: stop before the plate climbs it
                cn = self.env.cube_pos(cube)[axis]
                c_stall = c_stall + 1 if abs(cn - c_last) < 2e-4 else 0; c_last = cn
                if abs(cn - c0) > 0.002 and c_stall >= c_stall_n:
                    break
        for _ in range(6):          # relax: hold position with no push so the contact unloads before the plate leaves
            e = self.eef(); a = np.zeros(self.env.action_dim); a[:3] = np.clip(8.0 * (hold - e), -1, 1); a[axis] = 0.0; a[-1] = grip
            self.obs, _, _, _ = self.env.step(a); self.steps_used += 1
        return self.eef()

    def settle(self, steps=10):
        a = np.zeros(self.env.action_dim); a[-1] = -1
        for _ in range(steps):
            self.obs, _, _, _ = self.env.step(a)
        return self.env

    # ---- placement policy (expert with error; swapped for a learned policy in the notebook) ----
    def place(self, i, target_x, err_sd=0.0):
        e = self.env
        self.cur_i = i; self.cur_target = np.array([target_x, ROW_Y, CUBE_Z])
        c = e.cube_pos(i)
        self.goto([c[0], c[1], CUBE_Z + HOVER], -1, 40)
        self.goto([c[0], c[1], CUBE_Z + 0.003], -1, 35)
        self.goto([c[0], c[1], CUBE_Z + 0.003], +1, 12)          # close
        self.goto([c[0], c[1], CUBE_Z + HOVER], +1, 30)          # lift
        tx = target_x + err_sd * self.rng.normal()
        self.goto([tx, ROW_Y, CUBE_Z + HOVER], +1, 45)
        self.goto([tx, ROW_Y, CUBE_Z + 0.004], +1, 35)
        self.goto([tx, ROW_Y, CUBE_Z + 0.004], -1, 10)           # open
        self.goto([tx, ROW_Y, CUBE_Z + HOVER], -1, 25)
        self.settle(8)

    # ---- restoring organ ----
    def _seat_x(self, i, ref_face, pad_x, overdrive, z):
        c = self.env.cube_pos(i)
        self.goto([c[0] + CUBE_H + 0.04, c[1], CUBE_Z + HOVER], +1, 22)
        self.goto([c[0] + CUBE_H + 0.04, c[1], z], +1, 25)
        x_over = ref_face + PITCH + pad_x - overdrive
        self.push_axis(0, x_over, [x_over, c[1], z], +1, 80, amp=0.8)
        p = self.eef()
        self.goto([p[0] + 0.05, p[1], CUBE_Z + HOVER], +1, 20)

    def _seat_y(self, i, pad_y, overdrive, z):
        c = self.env.cube_pos(i)
        self.goto([c[0], c[1] + CUBE_H + 0.04, CUBE_Z + HOVER], +1, 22)
        self.goto([c[0], c[1] + CUBE_H + 0.04, z], +1, 25)
        y_over = RAIL_FACE + PITCH + pad_y - overdrive
        self.push_axis(1, y_over, [c[0], y_over, z], +1, 80, amp=0.8)
        p = self.eef()
        self.goto([p[0], p[1] + 0.05, CUBE_Z + HOVER], +1, 20)

    def push_diag(self, target_xy, z, grip, max_steps, amp=0.85, tol=0.002):
        """Saturated push toward a corner along both x and y at once. Stops when
        both axes have arrived or both are arrested by contact."""
        tx, ty = target_xy
        last = self.eef()[:2].copy(); stall = 0
        for _ in range(max_steps):
            e = self.eef()
            dx, dy = tx - e[0], ty - e[1]
            if abs(dx) < tol and abs(dy) < tol:
                break
            a = np.zeros(self.env.action_dim)
            a[0] = amp * np.sign(dx) if abs(dx) >= tol else 0.0
            a[1] = amp * np.sign(dy) if abs(dy) >= tol else 0.0
            a[2] = np.clip(8.0 * (z - e[2]), -1, 1)
            a[-1] = grip
            self._step(a)
            stall = stall + 1 if np.linalg.norm(e[:2] - last) < 2e-4 else 0
            last = e[:2].copy()
            if stall >= 8:
                break
        return self.eef()

    # ---- restoring organ v3: plate push into the corner, with re-pick recovery ----
    PAD_X = 0.010     # eef site -> closed-finger plate face along x (measured at contact onset)
    PAD_Y = 0.021     # same along y (the fingers' outer faces)
    PAD = PAD_X
    PUSH_Z = CUBE_Z           # plate site height: fingertips on the table
    Z_BIAS = -0.15            # mild downward action during pushes (the plate otherwise climbs the cube face; a hard press shoves cubes over the rail)

    def yaw(self):
        q = self.obs["robot0_eef_quat"]                       # xyzw
        x, y, z, w = q
        return float(np.arctan2(2 * (w * z + x * y), 1 - 2 * (y * y + z * z)))

    def home_pose(self, pos, quat, grip=-1, max_steps=60, tol_pos=0.004, tol_rot=0.02):
        """Drive the eef to a full pose (position + orientation). Used to return the robot to the exact
        state the policy was trained from: after a restore the wrist is 5-10 deg off in yaw AND roll,
        which the policy cannot see and which makes its next grasp miss."""
        pos = np.asarray(pos, float); qt = np.asarray(quat, float)
        for _ in range(max_steps):
            e = self.eef(); q = np.asarray(self.obs["robot0_eef_quat"], float)          # xyzw
            # rotation that takes q to qt, as a world-frame axis-angle
            x, y, z, w = q; qi = np.array([-x, -y, -z, w])                             # inverse
            ax, ay, az, aw = qt; bx, by, bz, bw = qi
            d = np.array([aw*bx + ax*bw + ay*bz - az*by, aw*by - ax*bz + ay*bw + az*bx,
                          aw*bz + ax*by - ay*bx + az*bw, aw*bw - ax*bx - ay*by - az*bz])   # qt * q^-1
            if d[3] < 0: d = -d
            ang = 2 * np.arccos(np.clip(d[3], -1, 1)); s = np.sqrt(max(1 - d[3]**2, 1e-12))
            rot = d[:3] / s * ang if ang > 1e-6 else np.zeros(3)
            perr = pos - e
            if np.linalg.norm(perr) < tol_pos and ang < tol_rot:
                break
            a = np.zeros(self.env.action_dim)
            a[:3] = np.clip(8.0 * perr, -1, 1); a[3:6] = np.clip(3.0 * rot, -1, 1); a[-1] = grip
            self._step(a)
        return self.eef()

    @staticmethod
    def _qmul(a, b):
        ax, ay, az, aw = a; bx, by, bz, bw = b        # xyzw
        return np.array([aw*bx + ax*bw + ay*bz - az*by, aw*by - ax*bz + ay*bw + az*bx,
                         aw*bz + ax*by - ay*bx + az*bw, aw*bw - ax*bx - ay*by - az*bz])

    def quat(self):
        return np.asarray(self.obs["robot0_eef_quat"], float).copy()

    def yawed(self, q, angle):
        """q rotated by `angle` about the world z axis."""
        return self._qmul(np.array([0.0, 0.0, np.sin(angle / 2), np.cos(angle / 2)]), q)

    def set_yaw(self, target, hold, grip, max_steps=45):
        hold = np.asarray(hold, float)
        for _ in range(max_steps):
            err = (target - self.yaw() + np.pi) % (2 * np.pi) - np.pi
            if abs(err) < 0.03:
                break
            a = np.zeros(self.env.action_dim)
            a[:3] = np.clip(8.0 * (hold - self.eef()), -1, 1)
            a[5] = float(np.clip(3.0 * err, -1, 1)); a[-1] = grip
            self._step(a)

    def cube_upright(self, i):
        m = np.array(self.env.sim.data.body_xmat[self.env.cube_bid[i]]).reshape(3, 3)
        return bool(np.max(np.abs(m[2])) > 0.95)          # some face is up (any yaw)

    def seated_ref(self, upto):
        """x of the rightmost cube among 0..upto-1 that is physically seated in the row (on the table,
        within 2.5 cm of the rail line); None if the row is empty -> the stop is the fence."""
        xs = [float(self.env.cube_pos(j)[0]) for j in range(upto)
              if abs(self.env.cube_pos(j)[1] - ROW_Y) < 0.025 and self.env.cube_pos(j)[2] < CUBE_Z + 0.008]
        return max(xs) if xs else None

    def in_zone(self, i, ref_face, fy=RAIL_FACE):
        """Pushable zone: on the table, in front of the y fence, not on/over the x fence, roughly upright, near its row."""
        c = self.env.cube_pos(i)
        return bool(c[2] < CUBE_Z + 0.008 and fy + CUBE_H - 0.006 < c[1] < fy + CUBE_H + 0.06   # near the row, not still on the staging row
                    and c[0] > ref_face + CUBE_H - 0.004 and self.cube_upright(i))

    def cube_yaw_mod90(self, i):
        m = np.array(self.env.sim.data.body_xmat[self.env.cube_bid[i]]).reshape(3, 3)
        # yaw of whichever horizontal face-normal is closest to +x, wrapped to (-45, 45] degrees
        ax = [m[:, j] for j in range(3) if abs(m[2, j]) < 0.5]     # horizontal body axes
        if not ax:                                                 # balanced on a corner: no axis is near-horizontal
            ax = [m[:, int(np.argmin(np.abs(m[2, :])))]]           # take the least vertical one
        best = max(ax, key=lambda v: abs(v[0]))
        y = np.arctan2(best[1], best[0])
        return float((y + np.pi / 4) % (np.pi / 2) - np.pi / 4)

    def push_off_top(self, i, max_steps=70):
        """Cube resting on top of the fence/predecessor: plate-push it in +x (the open side)
        until it drops onto the table. Plate rides 1 cm above the surface it sits on."""
        c = self.env.cube_pos(i); z = c[2] + 0.010
        ax = c[0] - CUBE_H - self.PAD - 0.02
        self.goto([ax, c[1], CUBE_Z + HOVER], +1, 20)
        self.goto([ax, c[1], z], +1, 22)
        for _ in range(max_steps):
            e = self.eef()
            a = np.zeros(self.env.action_dim)
            a[0] = 0.6; a[1] = float(np.clip(8.0 * (c[1] - e[1]), -1, 1)); a[2] = float(np.clip(8.0 * (z - e[2]), -1, 1)); a[-1] = +1
            self._step(a)
            if self.env.cube_pos(i)[2] < CUBE_Z + 0.008:
                break
        p = self.eef()
        self.goto([p[0], p[1], CUBE_Z + HOVER], -1, 18)
        self.settle(10)

    def repick(self, i, drop_xy, jaws="y"):
        """Recovery for a cube on the table but out of zone (behind the rail, beside the
        predecessor): grasp with the jaws along x, aligned to the cube's yaw, and drop it
        inside the pushable zone."""
        c = self.env.cube_pos(i); q0 = self.q_home
        hov = [c[0], c[1], CUBE_Z + HOVER]
        self.goto(hov, -1, 35)                                   # far reaches (staging row) track slowly: give the approach time
        if jaws == "x":                                     # beside the fence: jaws along x, aligned to the cube yaw
            self.home_pose(hov, self.yawed(q0, np.pi / 2 + self.cube_yaw_mod90(i)), -1)
        c = self.env.cube_pos(i)
        # grasp height: the normal expert height (tips at the table) unless a y-jaw finger would cross the rail;
        # on the staging row there is no rail, and a raised grasp only pinches the cube's top edges and slips
        near_rail = jaws == "y" and c[1] < ROW_Y + 0.06
        zg = max(c[2] + 0.008, TABLE_Z + 0.038) if near_rail else c[2] + 0.008
        self.goto([c[0], c[1], zg], -1, 40)
        c = self.env.cube_pos(i); self.goto([c[0], c[1], zg], -1, 12)   # re-centre on the cube before closing (an off-centre grasp slips in transit)
        self.goto([c[0], c[1], zg], +1, 12)
        self.goto([c[0], c[1], CUBE_Z + HOVER], +1, 30)
        self.goto([drop_xy[0], drop_xy[1], CUBE_Z + HOVER], +1, 45)
        self.goto([drop_xy[0], drop_xy[1], CUBE_Z + 0.008], +1, 35)
        self.goto([drop_xy[0], drop_xy[1], CUBE_Z + 0.008], -1, 10)
        hov = [drop_xy[0], drop_xy[1], CUBE_Z + HOVER]
        self.goto(hov, -1, 20)
        if jaws == "x":
            self.home_pose(hov, q0, -1)
        self.settle(5)

    def push_x_pressed(self, target, y_hold, z, max_steps, amp=0.8, press=0.25, tol=0.0015):
        """Saturated push along -x while pressing gently toward the rail, so the rail keeps
        the cube square as it slides. Stops on arrival or contact stall."""
        last = self.eef()[0]; stall = 0
        for _ in range(max_steps):
            e = self.eef(); d = target - e[0]
            if abs(d) < tol:
                break
            a = np.zeros(self.env.action_dim)
            a[0] = amp * np.sign(d)
            a[1] = -press
            a[2] = float(np.clip(8.0 * (z - e[2]), -1, 1)); a[-1] = +1
            self._step(a)
            stall = stall + 1 if abs(e[0] - last) < 1e-4 else 0
            last = e[0]
            if stall >= 14:
                break
        return self.eef()

    def restore(self, i, prev_x, overdrive=0.004, nominal_x=None, nominal_y=None):
        """Restoring organ, 3-D. The closed fingers form a 2 cm plate. Phase 0: if the cube
        is outside the pushable zone (on the fence, behind the rail, tipped), re-pick it into
        the zone. Phase 1: seat it against the rail (plate on its +y face). Phase 2: push it
        along the rail into the fence (stop or predecessor), pressing toward the rail so the
        rail squares it. Every contact is saturated, not P-controlled, and terminates on a
        stall. (The earlier grasp-and-slide version put one jaw on the rail and crept at
        0.2 mm/step: it could correct at most ~1 cm per call.)"""
        e = self.env
        # the fence comes from the scene geometry, not from task bookkeeping: the nearest pushable face in -x
        # at the nominal slot (stop, or whichever cube is seated to the left). Legacy callers without a nominal
        # fall back to the predecessor-based reference.
        ny = ROW_Y if nominal_y is None else nominal_y
        if nominal_x is not None:
            from rs_geom import find_fences
            f = find_fences(e, i, (nominal_x, ny), CUBE_H, CUBE_Z)
            ref_face = f["x"][0] if f["x"] is not None else STOP_FACE
            fy = f["y"][0] if f["y"] is not None else RAIL_FACE
        else:
            ref_face = STOP_FACE if prev_x is None else prev_x + CUBE_H; fy = RAIL_FACE
        seat_y = fy + CUBE_H                                     # canonical y: against the y fence
        # fence sanity check (diagnosed 2026-09-14): when the expected neighbour is missing or displaced, find_fences
        # returns the next face one cell further back (error exactly -40 mm) and the push seats the part into the hole.
        # A fence more than half a pitch short of the nominal cell is not the cell's fence: stop at the nominal instead.
        if nominal_y is not None and seat_y < nominal_y - 0.5 * PITCH:
            seat_y = nominal_y
            fy = seat_y - CUBE_H                                 # in_zone must use the corrected fence too
            self.fence_rejects = getattr(self, "fence_rejects", 0) + 1
        if nominal_x is not None and ref_face + CUBE_H < nominal_x - 0.5 * PITCH:
            ref_face = nominal_x - CUBE_H
            self.fence_rejects = getattr(self, "fence_rejects", 0) + 1
        if not self.in_zone(i, ref_face, fy):
            if e.cube_pos(i)[2] > CUBE_Z + 0.025:                 # sitting on top of another part
                # Lift it off rather than shoving it sideways. A shove only works when the free direction is
                # known (a 1-D row); in a 2-D block the open side depends on which cells are filled, and a fixed
                # +x shove rakes the finished work (4.4 cm of disturbance measured). A lift is direction-agnostic.
                self.repick(i, [ref_face + CUBE_H + 0.05, seat_y + 0.03], jaws="x")
            if not self.in_zone(i, ref_face, fy):                  # on the table but out of zone
                c = e.cube_pos(i)
                # staging row (far from the rail): y-jaws, since x-jaws collide with the next staged cube 2 cm away.
                # near the row: x-jaws, so the 3 cm rail is never under a finger.
                self.repick(i, [ref_face + CUBE_H + 0.05, seat_y + 0.03], jaws="y" if c[1] > seat_y + 0.06 else "x")
        c = e.cube_pos(i); z = self.PUSH_Z
        # phase 1: seat against the rail. The finger BACKS are wedge-shaped (a push with them lands high on the
        # cube, tilts it 17 deg into the rail and it springs back 7 mm on release), so rotate the jaws along x and
        # push with the flat finger SIDES, exactly the surface the x push uses.
        q0 = self.q_home; hov = [c[0], c[1] + CUBE_H + self.PAD_X + 0.025, CUBE_Z + HOVER]   # rotate relative to the STRAIGHT wrist, not the current (possibly bent) one
        self.goto(hov, +1, 20); self.home_pose(hov, self.yawed(q0, np.pi / 2), +1)   # full-pose: yaw-only control leaves pitch/roll errors at far reach
        self.goto(hov, +1, 30)                                   # the rotation displaces the site by ~3 cm: re-centre before descending
        ay = hov[1]
        cx = c[0] + getattr(self, "y_push_dx", 0.0)             # optional plate offset away from a finished neighbour column
        self.goto([cx, ay, z], +1, 30)
        y_t = seat_y + CUBE_H + self.PAD_X - 0.012                # eef backstop 1.2 cm past the seat; the cube-arrival test ends the push
        self.push_axis(1, y_t, [cx, y_t, z], +1, 60, amp=0.6, cube=i, z_bias=self.Z_BIAS, cube_target=seat_y + 0.003)
        if abs(self.cube_yaw_mod90(i)) > np.radians(5):
            p = self.eef(); self.goto([p[0], p[1] + 0.03, z], +1, 10)
            self.push_axis(1, y_t, [cx, y_t, z], +1, 40, amp=0.6, cube=i, c_stall_n=12, z_bias=self.Z_BIAS, cube_target=seat_y + 0.003)
        p = self.eef()
        self.goto([p[0], p[1] + 0.03, z], +1, 10)             # retreat BACK first at push height (a diagonal retreat drags the cube 1 cm)
        hov = [p[0], p[1] + 0.03, CUBE_Z + HOVER]
        self.goto(hov, +1, 15); self.home_pose(hov, q0, +1)                  # straight wrist before the x push
        # phase 2: push along the rail into the fence
        c = e.cube_pos(i); ax = c[0] + CUBE_H + self.PAD_X + 0.025
        self.goto([ax, c[1], CUBE_Z + HOVER], +1, 20)
        self.goto([ax, c[1], z], +1, 22)
        # target the nominal slot measured from the STOP (not the predecessor): the push seats the cube against
        # its predecessor and keeps pushing until the whole row is compacted against the stop, so a predecessor's
        # residual error is removed instead of inherited. Terminates on a genuine stall of the pushed cube.
        slot = ref_face + CUBE_H if nominal_x is None else min(nominal_x, ref_face + CUBE_H)
        x_t = slot + CUBE_H + self.PAD_X - 0.012                  # eef backstop; the cube-arrival test ends the push
        # y is P-held at the SEAT (a few mm past the cube centre): the sliding cube gets a gentle pull onto the rail
        # same mild hold-down as the rail push: a harder press (-0.35) loads the fingertips on the table and the long-row compaction stalls
        self.push_axis(0, x_t, [x_t, seat_y, z], +1, 130, amp=1.0, cube=i, c_stall_n=10, z_bias=self.Z_BIAS, cube_target=slot + 0.003)
        p = self.eef()
        self.goto([p[0] + 0.03, p[1], z], +1, 8)
        self.goto([p[0] + 0.03, p[1], CUBE_Z + HOVER], -1, 18)
        self.settle(8)

    def noop(self, steps=420):   # matched to the v7 restore's measured extra budget (~416 steps/stage without recovery)
        p = self.eef()
        q = steps // 4
        self.goto([p[0], p[1], CUBE_Z + HOVER + 0.05], -1, q)
        self.goto([p[0] + 0.04, p[1], CUBE_Z + HOVER + 0.05], -1, q)
        self.goto([p[0] - 0.04, p[1], CUBE_Z + HOVER + 0.06], -1, q)
        self.goto([p[0], p[1], CUBE_Z + HOVER + 0.05], -1, steps - 3 * q)
        self.settle(8)

    # ---- chain ----
    def run(self, n, mode="none", err_sd=0.0, place_fn=None):
        """place_fn(chain, i, target_x) may replace the expert (learned policy)."""
        self.obs = self.env.reset()
        devs, prev = [], None
        self.placed = []                      # cube pose right after placement, before restore/noop (diagnostics)
        for k in range(n):
            nominal = X0 + k * PITCH
            target = X0 if prev is None else prev + PITCH
            if place_fn is None:
                self.place(k, target, err_sd)
            else:
                place_fn(self, k, target)
            self.placed.append([round(float(v), 4) for v in self.env.cube_pos(k)])
            if mode == "restore":
                self.restore(k, prev, nominal_x=nominal)
            elif mode == "noop":
                self.noop()
            # settle clear before measuring, every mode
            p = self.eef(); self.goto([p[0], p[1], CUBE_Z + HOVER + 0.05], -1, 15); self.settle(15)
            prev = self.seated_ref(k + 1)
            devs.append(abs(float(self.env.cube_pos(k)[0]) - nominal))
        self.final = [round(abs(float(self.env.cube_pos(j)[0]) - (X0 + j * PITCH)), 4) for j in range(n)]   # final-row metric
        return np.array(devs)


In [ ]:
%%writefile rs_geom.py
"""Restore synthesis from scene geometry.

A restoring organ needs a physical reference to push against. Instead of telling the
restore which object is the fence for each stage (the stop for cube 0, the predecessor
for cube k, the rail for row 0, the cube below for row r), this module reads the
collision geometry of the scene and finds, for a given nominal pose, the nearest static
or seated face in the -x and -y directions that the part could be pushed against.
Everything the executor needs (which faces, where) comes out of the MuJoCo model; the
task code only says where the part is supposed to end up.

The same function therefore serves the 1-D line-up (finds stop / predecessor and rail)
and the 2-D packing block (finds left neighbour / stop and lower neighbour / rail).
"""
import numpy as np
import mujoco

BOX = mujoco.mjtGeom.mjGEOM_BOX


def _candidate_geoms(env, exclude_body):
    """box geoms that can act as fences: static fixtures and free cubes, never the robot or the table."""
    m, d = env.sim.model._model, env.sim.data._data
    out = []
    for g in range(m.ngeom):
        if m.geom_type[g] != BOX:
            continue
        b = m.geom_bodyid[g]
        if b == exclude_body:
            continue
        name = mujoco.mj_id2name(m, mujoco.mjtObj.mjOBJ_GEOM, g) or ""
        bname = mujoco.mj_id2name(m, mujoco.mjtObj.mjOBJ_BODY, b) or ""
        if "robot" in bname or "gripper" in bname or "mount" in bname or "table" in name:
            continue
        R = d.geom_xmat[g].reshape(3, 3); ext = np.abs(R) @ m.geom_size[g][:3]     # world AABB half-extents
        c = d.geom_xpos[g].copy()
        out.append((name, c - ext, c + ext))
    return out


def find_fences(env, cube_index, nominal_xy, half, z_center, reach=0.30, tol_overlap=0.01):
    """Nearest pushable faces in -x and -y for a part of half-size `half` whose nominal centre is
    `nominal_xy` at height `z_center`. Returns dict(x=(face_x, name) | None, y=(face_y, name) | None).
    A face qualifies if it is at or beyond the part's own face in that direction (so the part can be
    pushed onto it), within `reach`, and overlaps the part's extent on the other two axes."""
    m = env.sim.model._model
    body = m.geom_bodyid[env.sim.model.geom_name2id(f"cube{cube_index}_g0")]
    nx, ny = nominal_xy; zlo, zhi = z_center - half, z_center + half
    best = {"x": None, "y": None}
    for name, lo, hi in _candidate_geoms(env, body):
        if hi[2] - zlo < tol_overlap or zhi - lo[2] < tol_overlap:   # no vertical overlap (a stacked cube, a low kerb)
            continue
        # -x fence: the geom's +x face, left of the part's -x face, overlapping the part in y
        if hi[0] <= nx - half + 0.004 and nx - half - hi[0] <= reach and min(hi[1], ny + half) - max(lo[1], ny - half) >= tol_overlap:
            if best["x"] is None or hi[0] > best["x"][0]:
                best["x"] = (float(hi[0]), name)
        # -y fence: the geom's +y face, below the part's -y face, overlapping the part in x
        if hi[1] <= ny - half + 0.004 and ny - half - hi[1] <= reach and min(hi[0], nx + half) - max(lo[0], nx - half) >= tol_overlap:
            if best["y"] is None or hi[1] > best["y"][0]:
                best["y"] = (float(hi[1]), name)
    return best

def space_free(env, exclude_body, centre, half, tol=0.004):
    """Is the box of half-size `half` centred at `centre` (x, y, z) clear of every other part and
    fixture? Used to decide where a mis-stacked part can be shoved to, instead of assuming a task
    layout: in an open row the far side is free, in a packed block it is not."""
    c = np.asarray(centre, float); lo_q, hi_q = c - half + tol, c + half - tol
    for _, lo, hi in _candidate_geoms(env, exclude_body):
        if np.all(hi > lo_q) and np.all(lo < hi_q):
            return False
    return True


In [ ]:
%%writefile rs_pack.py
"""Gate 2 (second assembly topology): 2-D PACKING on the same rig.
Cubes fill a rows x columns block in row-major order. Cube (r, c) has TWO contact references:
in x the stop (c = 0) or the cube at (r, c-1); in y the rail (r = 0) or the cube at (r-1, c).
Errors therefore propagate along two axes and a failed cube blocks a whole row and column,
which is a different failure topology from the 1-D line-up. Same three arms, same restoring
organ (plate push into both references), same success rule (every cube within TOL of its
nominal cell, per stage and at the end)."""
import numpy as np, json, time, sys
from rs_chain import Chain, X0, ROW_Y, PITCH, CUBE_H, CUBE_Z, HOVER, STOP_FACE, RAIL_FACE, TABLE_Z
COLS = 4


def cell(k):
    return k // COLS, k % COLS                       # row, column


def nominal(k):
    r, c = cell(k)
    return X0 + c * PITCH, ROW_Y + r * PITCH


class PackChain(Chain):
    def seated_xy(self, j):
        p = self.env.cube_pos(j)
        return p if p[2] < CUBE_Z + 0.008 else None

    def refs(self, k):
        """physical reference faces for cube k: x-face of the left neighbour (or stop), y-face of the neighbour below (or rail)."""
        r, c = cell(k)
        fx = STOP_FACE
        if c > 0:
            p = self.seated_xy(k - 1)
            if p is not None and abs(p[1] - nominal(k - 1)[1]) < 0.025:
                fx = float(p[0]) + CUBE_H
        fy = RAIL_FACE
        if r > 0:
            p = self.seated_xy(k - COLS)
            if p is not None and abs(p[0] - nominal(k - COLS)[0]) < 0.025:
                fy = float(p[1]) + CUBE_H
        return fx, fy

    def place_cell(self, i, k, err_sd=0.0):
        e = self.env; c = e.cube_pos(i); nx, ny = nominal(k)
        self.goto([c[0], c[1], CUBE_Z + HOVER], -1, 40)
        self.goto([c[0], c[1], CUBE_Z + 0.003], -1, 35)
        self.goto([c[0], c[1], CUBE_Z + 0.003], +1, 12)
        self.goto([c[0], c[1], CUBE_Z + HOVER], +1, 30)
        tx = nx + err_sd * self.rng.normal(); ty = ny + err_sd * self.rng.normal()
        self.goto([tx, ty, CUBE_Z + HOVER], +1, 45)
        self.goto([tx, ty, CUBE_Z + 0.004], +1, 35)
        self.goto([tx, ty, CUBE_Z + 0.004], -1, 10)
        self.goto([tx, ty, CUBE_Z + HOVER], -1, 25)
        self.settle(8)

    def place_xy(self, i, tx, ty, err_sd=0.0):
        """single-stage expert placement at an explicit (x, y) target: used to record 2-D demos."""
        e = self.env; c = e.cube_pos(i)
        self.cur_i = i; self.cur_target = np.array([tx, ty, CUBE_Z])       # policy-facing target (recorder / rollout)
        self.goto([c[0], c[1], CUBE_Z + HOVER], -1, 40)
        self.goto([c[0], c[1], CUBE_Z + 0.003], -1, 35)
        self.goto([c[0], c[1], CUBE_Z + 0.003], +1, 12)
        self.goto([c[0], c[1], CUBE_Z + HOVER], +1, 30)
        tx = tx + err_sd * self.rng.normal(); ty = ty + err_sd * self.rng.normal()
        self.goto([tx, ty, CUBE_Z + HOVER], +1, 45)
        self.goto([tx, ty, CUBE_Z + 0.004], +1, 35)
        self.goto([tx, ty, CUBE_Z + 0.004], -1, 10)
        self.goto([tx, ty, CUBE_Z + HOVER], -1, 25)
        self.settle(8)

    def in_cell_zone(self, i, fx, fy):
        c = self.env.cube_pos(i)
        return bool(c[2] < CUBE_Z + 0.008 and c[0] > fx + CUBE_H - 0.004 and c[1] > fy + CUBE_H - 0.006
                    and c[1] < ROW_Y + 2 * PITCH + 0.06 and self.cube_upright(i))

    def restore_cell(self, i, k):
        """the line-up restore, unchanged: nominal cell in, fences from geometry."""
        nx, ny = nominal(k)
        self.restore(i, None, nominal_x=nx, nominal_y=ny)

    def dev(self, j):
        c = self.env.cube_pos(j); nx, ny = nominal(j)
        return float(np.hypot(c[0] - nx, c[1] - ny))

    def run(self, n, mode="none", err_sd=0.0, place_fn=None):
        self.obs = self.env.reset(); devs = []; self.placed = []
        for k in range(n):
            if place_fn is None:
                self.place_cell(k, k, err_sd)
            else:
                place_fn(self, k, nominal(k))
            self.placed.append([round(float(v), 4) for v in self.env.cube_pos(k)])
            if mode == "restore":
                self.restore_cell(k, k)
            elif mode == "noop":
                self.noop()
            p = self.eef(); self.goto([p[0], p[1], CUBE_Z + HOVER + 0.05], -1, 15); self.settle(15)
            devs.append(self.dev(k))
        self.final = [round(self.dev(j), 4) for j in range(n)]
        return np.array(devs)


if __name__ == "__main__":
    err_sd = float(sys.argv[1]) if len(sys.argv) > 1 else 0.02
    seeds = int(sys.argv[2]) if len(sys.argv) > 2 else 6
    TOL = 0.008; rows = []; ch = PackChain(n_cubes=8, seed=0); t0 = time.time()
    for mode in ("none", "noop", "restore"):
        for n in (2, 4, 8):
            for s in range(seeds):
                ch.reseed(900 + 37 * n + s); d = ch.run(n, mode=mode, err_sd=err_sd)
                rows.append(dict(mode=mode, n=n, seed=s, err_sd=err_sd, dev=d.tolist(), success=bool(np.all(d <= TOL)),
                                 final=ch.final, final_success=bool(all(v <= TOL for v in ch.final)),
                                 steps=ch.steps_used, placed=ch.placed))
                print(f"{mode:<8} N={n} s={s} devs={np.round(d,4)} ok={rows[-1]['success']} final_ok={rows[-1]['final_success']} {(time.time()-t0)/60:.1f}min", flush=True)
                json.dump(rows, open(f"rs_pack_{err_sd}.json", "w"))
    for mode in ("none", "noop", "restore"):
        srs = [np.mean([r["success"] for r in rows if r["mode"] == mode and r["n"] == n]) for n in (2, 4, 8)]
        fss = [np.mean([r["final_success"] for r in rows if r["mode"] == mode and r["n"] == n]) for n in (2, 4, 8)]
        print(f"{mode:<8} SR N=2/4/8 = {srs[0]:.2f}/{srs[1]:.2f}/{srs[2]:.2f}   final {fss[0]:.2f}/{fss[1]:.2f}/{fss[2]:.2f}")


In [ ]:
%%writefile rs_pack16.py
"""Run 3 rig: a 4 x 4 packing block (16 cells, pitch 0.04) so horizons up to N=16 stay inside the
workspace. Rows grow along +y from the rail, columns along +x from the stop (rs_pack.nominal with
COLS=4 already encodes this). Changes against the 2 x 4 rig, all fixtures only:
  * 16 cubes staged in TWO rows (y = STAGE_YS), 8 per row, x from STAGE_X0 in STAGE_DX steps;
  * the stop is lengthened in y so it spans all four rows (the 2 x 4 stop ends at y = 0.008, which
    leaves row 3 of column 0 with no x fence at all).
The restore is unchanged: PackChain.restore_cell -> Chain.restore with fences from rs_geom."""
import sys, time, json
import numpy as np
import rs_chain as _rc
from rs_chain import LineupEnv, CUBE_H, CUBE_Z, TABLE_Z, ROW_Y, PITCH, RAIL_Y, RAIL_HY, RAIL_HZ, STOP_X, STOP_HX
from rs_pack import PackChain, nominal, COLS
from robosuite.models.arenas import TableArena
from robosuite.models.objects import BoxObject
from robosuite.models.tasks import ManipulationTask
from robosuite.utils.mjcf_utils import array_to_string
from robosuite.environments.manipulation.manipulation_env import ManipulationEnv

ROWS = 4
N_CELLS = ROWS * COLS
STAGE_YS = (0.20, 0.30)          # 0.26 was tried: the open y-jaws land on the other staging row (grasp 18 mm high, far slots missed)
STAGE_X0, STAGE_DX, PER_ROW = -0.22, 0.06, 8
STOP_HY = 0.5 * ROWS * PITCH + 0.02                  # spans every row with 2 cm to spare
STOP_CY = ROW_Y + 0.5 * (ROWS - 1) * PITCH


def staging(i):
    return [STAGE_X0 + STAGE_DX * (i % PER_ROW), STAGE_YS[i // PER_ROW], CUBE_Z]


class Pack16Env(LineupEnv):
    def _load_model(self):
        ManipulationEnv._load_model(self)
        xpos = self.robots[0].robot_model.base_xpos_offset["table"](self.table_full_size[0])
        self.robots[0].robot_model.set_base_xpos(xpos)
        arena = TableArena(table_full_size=self.table_full_size, table_friction=self.table_friction,
                           table_offset=self.table_offset)
        arena.set_origin([0, 0, 0])
        self.cubes = [BoxObject(name=f"cube{i}", size=[CUBE_H] * 3, rgba=[0.85, 0.45, 0.2, 1])
                      for i in range(self.n_cubes)]
        self.rail = BoxObject(name="rail", size=[0.32, RAIL_HY, RAIL_HZ], rgba=[0.45, 0.5, 0.55, 1], joints=None)
        self.rail.get_obj().set("pos", array_to_string([0.0, RAIL_Y, TABLE_Z + RAIL_HZ]))
        self.stop = BoxObject(name="stop", size=[STOP_HX, STOP_HY, CUBE_H], rgba=[0.45, 0.5, 0.55, 1], joints=None)
        self.stop.get_obj().set("pos", array_to_string([STOP_X, STOP_CY, CUBE_Z]))
        self.model = ManipulationTask(mujoco_arena=arena, mujoco_robots=[self.robots[0].robot_model],
                                      mujoco_objects=self.cubes + [self.rail, self.stop])

    def _reset_internal(self):
        super()._reset_internal()                        # one-row staging from LineupEnv (qpos only, no stepping)
        for i in range(self.n_cubes):
            self.set_cube(i, staging(i))


class Pack16Chain(PackChain):
    # Measured 2026-09-12: row-3 restores in the full 4 x 4 block rake the block (the 4.2 cm plate's x push loads the
    # row below; the block shifts and springs back, 2-35 mm). Holding the plate 4 mm further +y was tried and made
    # rows 2-3 worse, so it was reverted. Rows 0-2 (12 cells) are the validated horizon; see run 3 notebook.
    def __init__(self, n_cubes=N_CELLS, seed=0):
        # Chain.__init__ builds rs_chain.LineupEnv by name; swap in the 4 x 4 env for the duration of the call
        orig = _rc.LineupEnv
        _rc.LineupEnv = Pack16Env
        try:
            super().__init__(n_cubes=n_cubes, seed=seed)
        finally:
            _rc.LineupEnv = orig


if __name__ == "__main__":
    # local rig validation with the scripted placer: N=16, restore vs none
    err_sd = float(sys.argv[1]) if len(sys.argv) > 1 else 0.004
    seeds = int(sys.argv[2]) if len(sys.argv) > 2 else 3
    modes = sys.argv[3].split(",") if len(sys.argv) > 3 else ["restore", "none"]
    NC = int(sys.argv[4]) if len(sys.argv) > 4 else N_CELLS      # chain length; 12 fills rows 0-2 only
    TOL = 0.008; ch = Pack16Chain(seed=0); t0 = time.time(); rows = []
    ch.obs = ch.env.reset()
    print("staging ok:", all(abs(ch.env.cube_pos(i)[2] - CUBE_Z) < 0.003 for i in range(N_CELLS)), flush=True)
    for s in range(seeds):
        for mode in modes:
            ch.reseed(4000 + s); d = ch.run(NC, mode=mode, err_sd=err_sd)
            fin = np.array(ch.final)
            rows.append(dict(mode=mode, seed=s, dev=d.tolist(), final=ch.final, steps=ch.steps_used, placed=ch.placed))
            print(f"{mode:<8} s={s} steps/stage={ch.steps_used/N_CELLS:.0f} ok={bool(np.all(d <= TOL))} final_ok={bool(np.all(fin <= TOL))} "
                  f"{(time.time()-t0)/60:.1f} min", flush=True)
            print("   dev   (mm, row-major):", " ".join(f"{1000*v:.1f}" for v in d), flush=True)
            print("   final (mm) as 4x4 grid, row 0 = rail:", flush=True)
            for r in range(ROWS):
                print("     row", r, " ".join(f"{1000*v:6.1f}" for v in fin[r * COLS:(r + 1) * COLS]), flush=True)
            json.dump(rows, open(f"_pack16_rig_{err_sd}_{'_'.join(modes)}.json", "w"))


In [ ]:
%%writefile rs_pack_guard.py
"""Packing restore that does not rake the finished block (fixes for the N=12 far-row collapse), tested locally.

Run 3 (pack3x4_v1) failed at N=12: far-row restores loaded the finished block and shoved it (28-70 mm). Modes:
  restore                 unchanged restore, row-major fill (the failing baseline)
  restore_guard           collateral guard: abort a push when a placed cube moves > 1.5 mm, then re-seat offenders.
                          Measured 0/3: aborts leave cubes unseated.
  restore_retry           guard + retreat and retry at lower force (amplitude x1.0, x0.7, x0.5), re-seating disturbed
                          neighbours corner-first between attempts; a last unguarded gentle push if still unseated
  restore_colmajor        push order that never loads a finished row: fill column by column (row-major leaves the
                          4.2 cm plate overhanging the finished row below by ~1 mm during every x push), and offset the
                          y-push plate 1.5 mm away from the finished column
  restore_colmajor_retry  both
Every stage logs `collateral`: the largest displacement of an already-placed cube caused by that stage.
"""
import json
import sys
import time

import numpy as np
from rs_chain import CUBE_Z, HOVER
from rs_pack import COLS, nominal
from rs_pack16 import Pack16Chain

TOL = 0.008
MAX_RESEAT = 2
BLOCK_ROWS = 3


def fill_order(name, n):
    if name == "col":
        order = [r * COLS + c for c in range(COLS) for r in range(BLOCK_ROWS)]
    else:
        order = list(range(BLOCK_ROWS * COLS))
    return order[:n]


class GuardPack16Chain(Pack16Chain):
    watch = None
    cell_of = {}

    def dev(self, j):
        c = self.env.cube_pos(j)
        nx, ny = nominal(self.cell_of.get(j, j))
        return float(np.hypot(c[0] - nx, c[1] - ny))

    def _positions(self, k):
        return {j: self.env.cube_pos(j).copy() for j in range(k)}

    def _reseat(self, k, ok_before):
        offenders = sorted((j for j in ok_before if self.dev(j) > TOL), key=lambda j: self.cell_of[j])   # corner first
        for j in offenders[:MAX_RESEAT]:
            self.watch = [x for x in range(k + 1) if x != j]
            self.restore_cell(j, self.cell_of[j])
            self.watch = None
        return offenders[:MAX_RESEAT]

    def restore_and_reseat(self, k, cell):
        """Plain restore of cube k, then re-seat (normal force, no abort guard, no retry) any finished cube this stage
        pushed out of tolerance, corner-most first. Diagnosed 2026-09-14: row-2 restores knock the row-1 end cube
        (cell r1c3) from 3.7 to 10.6-13 mm, and nothing ever re-seated it. Returns (max collateral, re-seated cubes)."""
        before = self._positions(k)
        ok_before = [j for j in range(k) if self.dev(j) <= TOL]
        self.restore_cell(k, cell)
        # restore approaches from +x and +y, so a cube whose +x or +y cell is already filled cannot be re-seated
        # without landing the plate on that neighbour: skip it (reviewer finding, 2026-09-14)
        filled = {self.cell_of.get(j, j) for j in range(k + 1)}

        def clear(j):
            c = self.cell_of.get(j, j)
            right = c + 1 if (c + 1) // COLS == c // COLS else None
            return right not in filled and (c + COLS) not in filled
        offenders = sorted((j for j in ok_before if self.dev(j) > TOL and clear(j)),
                           key=lambda j: self.cell_of.get(j, j))[:MAX_RESEAT]
        for j in offenders:
            self.restore_cell(j, self.cell_of.get(j, j))
        moved = max((float(np.linalg.norm(self.env.cube_pos(j)[:2] - p[:2])) for j, p in before.items()), default=0.0)
        return moved, offenders

    def restore_guarded(self, k, retry=False):
        before, cell = self._positions(k), self.cell_of[k]
        ok_before = {j for j in range(k) if self.dev(j) <= TOL}
        reseated = []
        try:
            for scale in ((1.0, 0.7, 0.5) if retry else (1.0,)):
                a0 = self.collateral_aborts
                self.amp_scale, self.watch = scale, list(range(k))
                self.restore_cell(k, cell)
                self.watch = None
                aborted = self.collateral_aborts > a0
                self.amp_scale = 0.7
                reseated += self._reseat(k, ok_before)
                if not aborted and self.dev(k) <= TOL:
                    break
            if retry and self.dev(k) > TOL:            # last resort: gentle, unguarded
                self.amp_scale = 0.5
                self.restore_cell(k, cell)
        finally:
            self.watch, self.amp_scale = None, 1.0
        moved = max((float(np.linalg.norm(self.env.cube_pos(j)[:2] - p[:2])) for j, p in before.items()), default=0.0)
        return moved, reseated

    def run(self, n, mode="none", err_sd=0.0, place_fn=None):
        order = fill_order("col" if "colmajor" in mode else "row", n)
        self.cell_of = {k: order[k] for k in range(n)}
        self.y_push_dx = 0.0015 if "colmajor" in mode else 0.0
        self.obs = self.env.reset(); devs = []; self.placed = []; self.collateral = []; self.reseated = []; self.trace = []
        self.collateral_aborts = 0
        for k in range(n):
            cell = order[k]
            if place_fn is None:
                self.place_cell(k, cell, err_sd)
            else:
                place_fn(self, k, nominal(cell))
            self.placed.append([round(float(v), 4) for v in self.env.cube_pos(k)])
            if mode == "restore_reseat":
                moved, offenders = self.restore_and_reseat(k, cell)
                self.collateral.append(moved)
                self.reseated.append(offenders)
            elif mode in ("restore", "restore_colmajor"):
                before = self._positions(k)
                self.restore_cell(k, cell)
                self.collateral.append(max((float(np.linalg.norm(self.env.cube_pos(j)[:2] - p[:2]))
                                            for j, p in before.items()), default=0.0))
                self.reseated.append([])
            elif mode in ("restore_guard", "restore_retry", "restore_colmajor_retry"):
                moved, res = self.restore_guarded(k, retry="retry" in mode)
                self.collateral.append(moved)
                self.reseated.append(res)
            elif mode == "noop":
                self.noop()
            p = self.eef(); self.goto([p[0], p[1], CUBE_Z + HOVER + 0.05], -1, 15); self.settle(15)
            devs.append(self.dev(k))
            self.trace.append([round(self.dev(j), 4) for j in range(k + 1)])   # every placed cube's error after stage k
        self.final = [round(self.dev(j), 4) for j in range(n)]
        return np.array(devs)


if __name__ == "__main__":
    err_sd = float(sys.argv[1]) if len(sys.argv) > 1 else 0.012
    seeds = int(sys.argv[2]) if len(sys.argv) > 2 else 3
    modes = sys.argv[3].split(",") if len(sys.argv) > 3 else ["restore", "restore_retry", "restore_colmajor"]
    NC = int(sys.argv[4]) if len(sys.argv) > 4 else 12
    tag = sys.argv[5] if len(sys.argv) > 5 else "run"
    ch = GuardPack16Chain(seed=0); rows = []; t0 = time.time()
    for s in range(seeds):
        for mode in modes:
            ch.reseed(5000 + s); d = ch.run(NC, mode=mode, err_sd=err_sd)
            fin = np.array(ch.final)
            rows.append(dict(mode=mode, seed=s, err_sd=err_sd, dev=d.tolist(), final=ch.final, collateral=ch.collateral,
                             reseated=ch.reseated, aborts=ch.collateral_aborts, steps=ch.steps_used,
                             success=bool(np.all(d <= TOL)), final_success=bool(np.all(fin <= TOL))))
            print(f"{mode:<24} s={s} ok={rows[-1]['success']!s:<5} final_ok={rows[-1]['final_success']!s:<5} "
                  f"stages<=8mm {int((d <= TOL).sum())}/{NC} final<=8mm {int((fin <= TOL).sum())}/{NC} "
                  f"max collateral {1000 * max(ch.collateral):.1f} mm aborts {ch.collateral_aborts} "
                  f"steps/stage {ch.steps_used / NC:.0f}  {(time.time() - t0) / 60:.1f} min", flush=True)
            print("   dev (mm):       ", " ".join(f"{1000 * v:.1f}" for v in d), flush=True)
            print("   final (mm):     ", " ".join(f"{1000 * v:.1f}" for v in fin), flush=True)
            print("   collateral (mm):", " ".join(f"{1000 * v:.1f}" for v in ch.collateral), flush=True)
            json.dump(rows, open(f"_pack_fix_{tag}_{err_sd}.json", "w"))
    for mode in modes:
        v = [r for r in rows if r["mode"] == mode]
        print(f"{mode:<24} per-stage ok {sum(r['success'] for r in v)}/{len(v)}  final ok {sum(r['final_success'] for r in v)}/{len(v)}  "
              f"stages<=8mm {np.mean([np.mean(np.array(r['dev']) <= TOL) for r in v]):.2f}  "
              f"final cubes<=8mm {np.mean([np.mean(np.array(r['final']) <= TOL) for r in v]):.2f}  "
              f"max collateral {1000 * max(c for r in v for c in r['collateral']):.1f} mm")


In [ ]:
# ---- demos: 300 single-stage placements, 2-D targets over the 3 x 4 block ----
import h5py, logging
logging.getLogger("robosuite_logs").setLevel(logging.ERROR)
from rs_chain import CUBE_Z, ROW_Y, PITCH, X0
from rs_pack import COLS
from rs_pack16 import Pack16Chain, staging, N_CELLS, ROWS
BLOCK_ROWS = 3    # rows 0-2 of the 4 x 4 rig: fourth-row restores rake the block (measured locally)
DEMO = f"{OUT}/demos_pack3x4.hdf5"; N_DEMO = 300
LOCAL_DEMO = f"{RES}/demos_pack3x4.hdf5"
X_RANGE = (X0 - 0.01, X0 + (COLS - 1) * PITCH + 0.02)          # block columns plus margin
Y_RANGE = (ROW_Y - 0.01, ROW_Y + (BLOCK_ROWS - 1) * PITCH + 0.02)    # the three block rows plus margin
def demos_valid(p):
    try:
        with h5py.File(p, "r") as f:
            return len(f["data"].keys()) == N_DEMO and int(f["data"].attrs["total"]) > 0
    except Exception:
        return False
if demos_valid(DEMO):
    print("demos complete on Drive, skipping:", DEMO)
else:
    t0 = time.time(); rng = np.random.default_rng(0); errs = []
    dch = Pack16Chain(seed=1000)
    with h5py.File(LOCAL_DEMO, "w") as f:                       # written locally, copied to Drive when complete
        g = f.create_group("data"); total = 0
        for d in range(N_DEMO):
            dch.obs = dch.env.reset()
            # spawn at one of the 16 real staging slots (both staging rows), swapping cube 0 with that slot's cube,
            # so the demo start states cover exactly what the chain will present
            j = int(rng.integers(N_CELLS)); jit = rng.uniform(-0.005, 0.005, 2); sp = staging(j)
            if j != 0:
                dch.env.set_cube(j, staging(0))
            dch.env.set_cube(0, [sp[0] + jit[0], sp[1] + jit[1], CUBE_Z]); dch.settle(3)
            tx = float(rng.uniform(*X_RANGE)); ty = float(rng.uniform(*Y_RANGE))
            dch.rec = []; dch.place_xy(0, tx, ty); rec = dch.rec; dch.rec = None
            c = dch.env.cube_pos(0); errs.append(float(np.hypot(c[0] - tx, c[1] - ty)))
            obs = {k: np.stack([o[k] for o, _ in rec]) for k in rec[0][0]}
            acts = np.stack([a for _, a in rec]); T = len(acts)
            dg = g.create_group(f"demo_{d}"); dg.attrs["num_samples"] = T
            dg.create_dataset("actions", data=acts)
            dg.create_dataset("rewards", data=np.zeros(T, np.float32))
            dg.create_dataset("dones", data=np.zeros(T, np.int64))
            for k, v in obs.items():
                dg.create_dataset(f"obs/{k}", data=v)
            total += T
            if d % 25 == 0:
                print(f"  demo {d}: slot {j} T={T} expert err={errs[-1]:.4f}  {(time.time()-t0)/60:.1f} min", flush=True)
        g.attrs["total"] = total
        g.attrs["env_args"] = json.dumps({"env_name": "Pack3x4", "type": 1, "env_kwargs": {}})
    assert drive_ok(), "Drive dropped during demo collection: re-mount (cell 2) and re-run this cell"
    shutil.copyfile(LOCAL_DEMO, DEMO)
    assert demos_valid(DEMO), "copied demo file failed validation"
    print(f"wrote {N_DEMO} demos, {total} samples, expert err mean {np.mean(errs):.4f} max {np.max(errs):.4f}, {(time.time()-t0)/60:.1f} min")

In [ ]:
# ---- 2. train a robomimic diffusion policy on the demos ----
import robomimic, glob, re
import shutil
EPOCHS = 200
def _ckpts():
    c = sorted(glob.glob(f"{OUT}/dp_pack3x4/**/models/*.pth", recursive=True), key=lambda p: int(re.search(r"epoch_(\d+)", p).group(1)))   # numeric: plain sort puts epoch_50 after epoch_200
    return c, (int(re.search(r"epoch_(\d+)", c[-1]).group(1)) if c else 0)
CKPT, LAST_EP = _ckpts()
# Require the FINAL epoch, not merely some checkpoint: an interrupted run leaves epoch 50/100 on Drive and a
# "skip if any checkpoint exists" test then silently evaluates a half-trained policy.
if LAST_EP >= EPOCHS:
    print("training complete, skipping:", CKPT[-1])
else:
    if CKPT:
        print(f"only epoch {LAST_EP} of {EPOCHS} on Drive: training was interrupted, restarting clean")
        assert drive_ok(), 'Drive not mounted: refusing to touch dp_pack3x4'
        shutil.rmtree(f"{OUT}/dp_pack3x4", ignore_errors=True)
    tmpl = pathlib.Path(robomimic.__file__).parent / "exps/templates/diffusion_policy.json"
    cfg = json.load(open(tmpl))
    cfg["experiment"]["name"] = "dp_pack3x4"
    cfg["experiment"]["rollout"]["enabled"] = False          # we evaluate in our own chain
    cfg["experiment"]["save"]["enabled"] = True
    cfg["experiment"]["save"]["every_n_epochs"] = EPOCHS    # final checkpoint only: each is ~1 GB
    cfg["experiment"]["validate"] = False
    cfg["experiment"]["logging"]["log_wandb"] = False
    cfg["train"]["data"] = DEMO
    cfg["train"]["output_dir"] = f"{OUT}/dp_pack3x4"
    cfg["train"]["num_epochs"] = EPOCHS
    cfg["train"]["batch_size"] = 256
    cfg["train"]["hdf5_cache_mode"] = "all"
    cfg["observation"]["modalities"]["obs"]["low_dim"] = ["eef_pos", "gripper_qpos", "cube_pos", "target"]
    cfg["observation"]["modalities"]["obs"]["rgb"] = []
    cfgp = f"{OUT}/dp_cfg_pack3x4.json"; json.dump(cfg, open(cfgp, "w"), indent=1)
    t0 = time.time()
    r = subprocess.run([sys.executable, "-m", "robomimic.scripts.train", "--config", cfgp],
                       capture_output=True, text=True, input="y\n")   # answers robomimic's "overwrite? (y/n)" if the run dir exists
    print(r.stdout[-3000:]); print(r.stderr[-2000:])
    CKPT, LAST_EP = _ckpts()
    print(f"trained in {(time.time()-t0)/60:.1f} min; reached epoch {LAST_EP}; checkpoints:", CKPT[-3:])
assert LAST_EP >= EPOCHS, f"training stopped at epoch {LAST_EP}, need {EPOCHS} -- read the log above"

In [ ]:
# ---- load the diffusion policy (DDIM-10 sampling), define the learned placement ----
import torch, glob, re, logging
from collections import deque
from robomimic.utils.file_utils import policy_from_checkpoint
logging.getLogger("robosuite_logs").setLevel(logging.ERROR)
from rs_chain import Chain, ROW_Y, CUBE_Z, PITCH, X0
from rs_pack import nominal
from rs_pack16 import Pack16Chain, N_CELLS
from rs_pack_guard import GuardPack16Chain
EPOCHS = 200
CK_DIR = f"{OUT}/dp_pack3x4"
CKPT = sorted(glob.glob(f"{CK_DIR}/**/models/*.pth", recursive=True), key=lambda p: int(re.search(r"epoch_(\d+)", p).group(1)))
LAST_EP = int(re.search(r"epoch_(\d+)", CKPT[-1]).group(1)) if CKPT else 0
assert LAST_EP >= EPOCHS, f"need the final epoch {EPOCHS} checkpoint under {CK_DIR}, found epoch {LAST_EP}"
print("using checkpoint:", CKPT[-1])
ck = torch.load(CKPT[-1], map_location="cpu", weights_only=False)
_cfg = json.loads(ck["config"]) if isinstance(ck["config"], str) else ck["config"]
_cfg["algo"]["ddpm"]["enabled"] = False                 # DDIM-10 at inference (7x faster, weights unchanged)
_cfg["algo"]["ddim"]["enabled"] = True
_cfg["algo"]["ddim"]["num_inference_timesteps"] = 10
ck["config"] = json.dumps(_cfg)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
policy, _ = policy_from_checkpoint(ckpt_dict=ck, device=DEVICE, verbose=False)
FRAME_STACK = int(_cfg["train"].get("frame_stack", 1))
T_POLICY = 240
print("frame_stack =", FRAME_STACK, "| sampler = DDIM-10 | device =", DEVICE)

ch = GuardPack16Chain(seed=7)                                     # ONE env for the whole notebook (robosuite leaks envs)
ch.obs = ch.env.reset()
HOME = ch.eef().copy(); HOME_QUAT = np.array(ch.obs["robot0_eef_quat"], float).copy()   # every demo started here
print("home eef:", np.round(HOME, 3), "quat:", np.round(HOME_QUAT, 3))

def learned_place(chain, i, target, home=True):
    # home=True: full-pose homing first (position AND orientation), the policy's training start state.
    # home=False is ONLY for the no-robot-reset arm.
    if home:
        p = chain.eef(); chain.goto([p[0], p[1], HOME[2]], -1, 15)
        chain.home_pose(HOME, HOME_QUAT, -1)
    if isinstance(target, (tuple, list, np.ndarray)):
        tx, ty = float(target[0]), float(target[1])
    else:
        tx, ty = float(target), ROW_Y
    chain.cur_i = i; chain.cur_target = np.array([tx, ty, CUBE_Z]); policy.start_episode()
    hist = deque([chain.obs_dict()] * FRAME_STACK, maxlen=FRAME_STACK)
    for _ in range(T_POLICY):
        ob = {k: np.stack([h[k] for h in hist]) for k in hist[0]} if FRAME_STACK > 1 else hist[-1]
        a = np.clip(np.asarray(policy(ob=ob)), -1, 1)
        chain._step(a)
        hist.append(chain.obs_dict())
    chain.settle(8)

def landing_dev(placed, k):          # packing metric: planar distance to the nominal cell
    nx, ny = nominal(k); return float(np.hypot(placed[0] - nx, placed[1] - ny))

# single-stage probe: the learned policy's own error sets the regime
errs = []
for k, tgt in [(k, tuple(nominal(k))) for k in (0, 5, 11)]:
    ch.obs = ch.env.reset(); learned_place(ch, k, tgt)
    c = ch.env.cube_pos(k); t = tgt if isinstance(tgt, (tuple, list)) else (tgt, ROW_Y)
    errs.append(float(np.hypot(c[0] - t[0], c[1] - t[1])))
print("learned single-stage |error| (m):", " ".join(f"{v:.4f}" for v in errs), f"| mean {np.mean(errs):.4f}")

In [ ]:
# ---- sweep: resumable per chain, keyed on (arm, N, seed) and the sampler TAG ----
N_LIST = [2, 4, 8, 12]; SEEDS = 6; TOL = 0.008
ARMS = {"restore_v2": ("restore", True), "restore_reseat": ("restore_reseat", True)}            # arm -> (chain mode, full-pose homing before each learned placement)
TAG = "pack3x4_v2"; RESULTS = "chain_pack3x4_v2.json"   # a NEW tag and file: rows from older runs are ignored, nothing is deleted
EST_S_PER_STAGE = {"restore_v2": 20.0, "restore_reseat": 23.0}  # T4 seconds per stage, calibrated on the v7 line-up run (90 chains, 83 min)
rows = [r for r in (load(RESULTS) or []) if r.get("sampler") == TAG]
done = {(r["mode"], r["n"], r["seed"]) for r in rows}
# ordered by (N, seed) with every arm inside: an interrupted sweep still holds PAIRED comparisons
todo = [(n, s, arm) for n in N_LIST for s in range(SEEDS) for arm in ARMS if (arm, n, s) not in done]
est = sum(n * EST_S_PER_STAGE[arm] for n, _, arm in todo) / 60
print(f"tag {TAG}: {len(rows)} chains on file, {len(todo)} to run, estimated {est:.0f} min ({est/60:.1f} h) on a T4")
t0 = time.time(); stages_done = 0; stages_total = sum(n for n, _, _ in todo)
for n, s, arm in todo:
    mode, home = ARMS[arm]
    ch.reseed(500 + 17 * n + s)
    torch.manual_seed(100000 + 1000 * n + s); np.random.seed(100000 + 1000 * n + s)   # paired: same sampler seed in every arm
    ch.fence_rejects = 0
    d = ch.run(n, mode=mode, place_fn=lambda c, i, t, _h=home: learned_place(c, i, t, home=_h))
    rows.append(dict(mode=arm, chain_mode=mode, home=home, n=n, seed=s, dev=[float(v) for v in d], placed=ch.placed,
                     final=ch.final, success=bool(np.all(d <= TOL)), final_success=bool(all(v <= TOL for v in ch.final)),
                     steps=ch.steps_used, sampler=TAG, collateral=getattr(ch, "collateral", None),
                     reseated=getattr(ch, "reseated", None), fence_rejects=ch.fence_rejects))
    save(RESULTS, rows)
    stages_done += n; el = (time.time() - t0) / 60
    eta = el / stages_done * (stages_total - stages_done)
    print(f"{arm:<12} N={n:<2} seed={s} ok={rows[-1]['success']} final_ok={rows[-1]['final_success']} "
          f"dev(mm)={' '.join(f'{1000*v:.0f}' for v in d)} | {el:.0f} min, ~{eta:.0f} min left", flush=True)
print("sweep complete:", len(rows), "chains for tag", TAG)

In [ ]:
# ---- analysis: success per arm, censoring-aware per-stage bounds, paired contrasts ----
from scipy.stats import beta as _beta, binomtest
rows = [r for r in (load(RESULTS) or []) if r.get("sampler") == TAG]
ARM_NAMES = [a for a in ARMS if any(r["mode"] == a for r in rows)]
def sel(arm, n): return [r for r in rows if r["mode"] == arm and r["n"] == n]
print(f"{'arm':<12}" + "".join(f"   N={n:<5}" for n in N_LIST) + "  steps/stage")
for a in ARM_NAMES:
    cells = []
    for n in N_LIST:
        rr = sel(a, n); cells.append(f"{sum(r['success'] for r in rr)}/{len(rr)}".rjust(8) + "  ")
    st = np.mean([r["steps"] / r["n"] for r in rows if r["mode"] == a])
    print(f"{a:<12}" + " ".join(cells) + f"  {st:7.0f}")
print("\nfinal success (every cube within tol at the END of the chain):")
for a in ARM_NAMES:
    print(f"  {a:<12}" + "  ".join(f"N={n}: {np.mean([r['final_success'] for r in sel(a, n)]) if sel(a, n) else float('nan'):.2f}" for n in N_LIST))

# censoring-aware per-stage reliability: Clopper-Pearson on observed successes, to the 1/N power, tightest over N
def _cp(k, n, upper, a=0.05):
    if upper: return 1.0 if k == n else float(_beta.ppf(1 - a, k + 1, n - k))
    return 0.0 if k == 0 else float(_beta.ppf(a, k, n - k + 1))
def bounds(arm):
    ks = {n: (int(sum(r["success"] for r in sel(arm, n))), len(sel(arm, n))) for n in N_LIST if sel(arm, n)}
    hi = min(_cp(k, m, True) ** (1 / n) for n, (k, m) in ks.items())
    lo = max(_cp(k, m, False) ** (1 / n) for n, (k, m) in ks.items())
    return lo, hi
print("\nper-stage reliability bounds (95% one-sided, tightest over N):")
B = {a: bounds(a) for a in ARM_NAMES}
for a in ARM_NAMES:
    print(f"  {a:<12} >= {B[a][0]:.3f}   <= {B[a][1]:.3f}")
if "restore" in B:
    others = [a for a in ARM_NAMES if a != "restore"]
    ok = all(B["restore"][0] > B[a][1] for a in others)
    print("\nACCEPT: restore's lower bound exceeds the upper bound of every other arm" if ok and others
          else "\nNOT ACCEPTED on the bounds test (report as is)")

# paired contrasts: same (N, seed) in both arms
CONTRASTS = [("restore_reseat", "restore_v2", "re-seat pass vs fence fix alone")]
def _mdev(r): return float(np.mean(r["dev"]))
summary = {"tag": TAG, "success": {a: {str(n): [int(sum(r["success"] for r in sel(a, n))), len(sel(a, n))] for n in N_LIST} for a in ARM_NAMES},
           "bounds": B, "contrasts": {}}
print("\npaired contrasts (A minus B; exact McNemar p on discordant pairs pooled over N; 95% bootstrap CI over pairs):")
rng = np.random.default_rng(0)
for A_, B_, what in CONTRASTS:
    if A_ not in ARM_NAMES or B_ not in ARM_NAMES:
        print(f"  {A_} vs {B_}: missing arm, skipped"); continue
    ia = {(r["n"], r["seed"]): r for r in rows if r["mode"] == A_}; ib = {(r["n"], r["seed"]): r for r in rows if r["mode"] == B_}
    keys = sorted(set(ia) & set(ib))
    if not keys:
        print(f"  {A_} vs {B_}: no paired chains yet"); continue
    ds = np.array([int(ia[k]["success"]) - int(ib[k]["success"]) for k in keys], float)
    dd = np.array([_mdev(ia[k]) - _mdev(ib[k]) for k in keys], float)
    b = int((ds > 0).sum()); c = int((ds < 0).sum())
    p = binomtest(b, b + c, 0.5).pvalue if b + c > 0 else 1.0
    bs = [rng.choice(len(keys), len(keys), replace=True) for _ in range(2000)]
    ci_s = np.percentile([ds[i].mean() for i in bs], [2.5, 97.5]); ci_d = np.percentile([dd[i].mean() for i in bs], [2.5, 97.5])
    per_n = "  ".join(f"N={n}: {np.mean([ds[j] for j, k in enumerate(keys) if k[0] == n]):+.2f}" for n in N_LIST if any(k[0] == n for k in keys))
    print(f"  {what}\n    {A_} - {B_}: success {ds.mean():+.2f} [{ci_s[0]:+.2f},{ci_s[1]:+.2f}]  ({per_n})  discordant {b}:{c}  p={p:.3g}"
          f"\n    mean per-chain deviation {1000*dd.mean():+.1f} mm [{1000*ci_d[0]:+.1f},{1000*ci_d[1]:+.1f}]  over {len(keys)} pairs")
    summary["contrasts"][f"{A_}-{B_}"] = dict(what=what, d_success=float(ds.mean()), ci=ci_s.tolist(), discordant=[b, c], p=float(p),
                                              d_dev_mm=float(1000 * dd.mean()), ci_dev_mm=(1000 * ci_d).tolist(), pairs=len(keys))
save(RESULTS.replace(".json", "_summary.json"), summary)

In [ ]:
# ---- error recursion: e_{k+1} = a * e_k + b on the longest chains, per arm ----
# Fit on all consecutive stage pairs pooled over seeds (least squares); CIs by bootstrapping SEEDS (whole chains).
# Fixed point b / (1 - a) exists only when |a| < 1. Claim under test: restore imposes a low fixed point (a near 0,
# small b); the bare chain accumulates (a near 1, or a large b).
# Caveat: along a chain this regression reports a near 1 for ANY smooth ramp, so read it together with the
# within-stage contraction below (deviation after the stage / deviation right after the placement).
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
rows = [r for r in (load(RESULTS) or []) if r.get("sampler") == TAG]
NREC = max(N_LIST)
ARM_NAMES = [a for a in ARMS if any(r["mode"] == a and r["n"] == NREC for r in rows)]
def chains(arm): return [np.asarray(r["dev"], float) for r in rows if r["mode"] == arm and r["n"] == NREC]
def fit(ch_list):
    x = np.concatenate([d[:-1] for d in ch_list]) if ch_list else np.zeros(0)
    y = np.concatenate([d[1:] for d in ch_list]) if ch_list else np.zeros(0)
    if len(x) < 2 or np.ptp(x) == 0:
        return float("nan"), float("nan"), float("nan")
    a, b = np.linalg.lstsq(np.c_[x, np.ones_like(x)], y, rcond=None)[0]
    fp = b / (1 - a) if abs(a) < 1 else float("nan")
    return float(a), float(b), float(fp)
rec = {}; rng = np.random.default_rng(1)
print(f"error recursion on N={NREC} chains (deviations in mm):")
print(f"  {'arm':<12} {'a [95% CI]':<24} {'b mm [95% CI]':<24} {'fixed point mm [95% CI]':<28} P(|a|<1)  seeds")
for arm in ARM_NAMES:
    cl = chains(arm); a, b, fp = fit(cl)
    boot = [fit([cl[i] for i in rng.integers(0, len(cl), len(cl))]) for _ in range(2000)]
    A = np.array([v[0] for v in boot]); Bv = np.array([v[1] for v in boot]); F = np.array([v[2] for v in boot])
    def ci(v, s=1.0):
        v = v[np.isfinite(v)]
        return (s * np.percentile(v, 2.5), s * np.percentile(v, 97.5)) if len(v) else (float("nan"), float("nan"))
    ca, cb, cf = ci(A), ci(Bv, 1000), ci(F, 1000)
    frac = float(np.mean(np.abs(A[np.isfinite(A)]) < 1)) if np.isfinite(A).any() else float("nan")
    print(f"  {arm:<12} {a:+.3f} [{ca[0]:+.3f},{ca[1]:+.3f}]   {1000*b:6.2f} [{cb[0]:6.2f},{cb[1]:6.2f}]   "
          f"{1000*fp:7.2f} [{cf[0]:7.2f},{cf[1]:7.2f}]      {frac:.2f}     {len(cl)}")
    rec[arm] = dict(a=a, a_ci=list(ca), b_mm=1000 * b, b_ci_mm=list(cb), fixed_point_mm=1000 * fp, fp_ci_mm=list(cf), p_abs_a_lt_1=frac, seeds=len(cl))

print(f"\nmean deviation per stage (mm), N={NREC}:")
print(f"  {'arm':<12}" + "".join(f"{k+1:>7}" for k in range(NREC)))
for arm in ARM_NAMES:
    m = np.mean(np.stack(chains(arm)), axis=0) if chains(arm) else np.full(NREC, np.nan)
    print(f"  {arm:<12}" + "".join(f"{1000*v:7.1f}" for v in m)); rec[arm]["mean_dev_mm"] = (1000 * m).tolist()

print("\nwithin-stage contraction, median over stages of dev(after stage) / dev(landing), landings > 2 mm:")
for arm in ARM_NAMES:
    ratios = [r["dev"][k] / landing_dev(r["placed"][k], k) for r in rows if r["mode"] == arm and r["n"] == NREC
              for k in range(NREC) if landing_dev(r["placed"][k], k) > 0.002]
    print(f"  {arm:<12} {np.median(ratios) if ratios else float('nan'):.3f}   ({len(ratios)} stages)")
    rec[arm]["within_stage_median_ratio"] = float(np.median(ratios)) if ratios else None

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
cols = plt.rcParams["axes.prop_cycle"].by_key()["color"]
for j, arm in enumerate(ARM_NAMES):
    cl = chains(arm)
    x = np.concatenate([d[:-1] for d in cl]); y = np.concatenate([d[1:] for d in cl])
    ax[0].scatter(1000 * np.maximum(x, 1e-4), 1000 * np.maximum(y, 1e-4), s=12, alpha=0.5, color=cols[j % len(cols)], label=arm)
    if np.isfinite(rec[arm]["a"]):
        g = np.linspace(max(x.min(), 1e-4), max(x.max(), 2e-4), 50)
        ax[0].plot(1000 * g, 1000 * np.maximum(rec[arm]["a"] * g + rec[arm]["b_mm"] / 1000, 1e-4), color=cols[j % len(cols)], lw=1.5)
    ax[1].plot(range(1, NREC + 1), rec[arm]["mean_dev_mm"], "o-", color=cols[j % len(cols)], label=arm)
lim = [0.1, 400]
ax[0].plot(lim, lim, "k--", lw=0.8, label="e(k+1) = e(k)")
ax[0].set_xscale("log"); ax[0].set_yscale("log"); ax[0].set_xlabel("e_k (mm)"); ax[0].set_ylabel("e_(k+1) (mm)")
ax[0].set_title(f"error recursion, N={NREC}"); ax[0].legend(fontsize=8)
ax[1].axhline(1000 * TOL, color="k", ls=":", lw=0.8, label="tolerance")
ax[1].set_yscale("log"); ax[1].set_xlabel("stage"); ax[1].set_ylabel("mean deviation (mm)")
ax[1].set_title("mean deviation vs stage"); ax[1].legend(fontsize=8)
fig.tight_layout()
PNG = RESULTS.replace(".json", "_recursion.png")
fig.savefig(f"{RES}/{PNG}", dpi=150); plt.close(fig)
print("\nfigure:", f"{RES}/{PNG}", "| mirrored to Drive:", mirror(PNG))
save(RESULTS.replace(".json", "_recursion.json"), rec)

In [ ]:
# ---- paired comparison with run 3 (same policy, same chain and sampler seeds): final-block success ----
v1 = [r for r in (load("chain_pack3x4_v1.json") or []) if r.get("sampler") == "pack3x4_v1"]
v2 = [r for r in (load(RESULTS) or []) if r.get("sampler") == TAG]
print(f"{'arm':<16}" + "".join(f"  N={n:<5}" for n in N_LIST) + "   final-block success")
for src, arm in [(v1, "none"), (v1, "noop"), (v1, "restore"), (v2, "restore_v2"), (v2, "restore_reseat")]:
    cells = []
    for n in N_LIST:
        sel = [r for r in src if r["mode"] == arm and r["n"] == n]
        cells.append(f"{sum(r['final_success'] for r in sel)}/{len(sel)}" if sel else "-")
    print(f"{arm:<16}" + "".join(f"  {c:<7}" for c in cells))

In [ ]:
# ---- save and shut down: mirror every result to Drive, write a manifest, flush ----
rows = [r for r in (load(RESULTS) or []) if r.get("sampler") == TAG]
mirrored = {p.name: mirror(p.name) for p in sorted(pathlib.Path(RES).iterdir()) if p.is_file() and not p.name.endswith(".tmp")}
manifest = {"saved_at": time.strftime("%Y-%m-%d %H:%M"), "tag": TAG, "results": RESULTS, "chains": len(rows),
            "expected_chains": len(N_LIST) * SEEDS * len(ARMS), "checkpoint": CKPT[-1] if CKPT else None, "mirrored": mirrored}
with open(f"{RES}/manifest_pack3x4_v2.json", "w") as f:
    json.dump(manifest, f, indent=1)
mirror("manifest_pack3x4_v2.json")
print(json.dumps(manifest, indent=1))
if manifest["chains"] < manifest["expected_chains"]:
    print("sweep INCOMPLETE: run the notebook top to bottom again, finished chains are skipped")
from google.colab import drive
drive.flush_and_unmount(); print("Drive flushed, safe to terminate.")